# Using SuperNeuroNAT to Code Logical Operations (XOR Gate)

When getting started using SuperNeuroMAT, it is useful to analyze basic problems and solve them using the neuron functions within SuperNeuroMAT that you will use later to complete more complex projects. Continuing with our logical operations, we can make the exclusive gates, XOR and XNOR, using SuperNeuroMAT.

For those unfamiliar with these operations, they are defined as:
<br>XOR Gate: Given a set of two inputs, return true if ONLY ONE of the inputs is true.
<br>XNOR gate: Given a set of two inputs, returns true if either NONE or BOTH of the inputs are true.

In terms of application in the world of coding, here are the truth tables for each gate given x and y. 1 designates true, 0 designates false.

<br> For the XOR gate given two inputs:
<br>![diagram of XOR_gate](img/logic_gates_images/xor_gate_diagram.png "XOR_gate")
<br> For the XNOR gate given two inputs:
<br>![diagram of XNOR_gate](img/logic_gates_images/xnor_gate_diagram.png "XNOR_gate")

For this tutorial, we will focus on the XOR gate specifically.

In [42]:
### First, install and import superneuromat
#  use [pip install superneuromat inside of your terminal]
import superneuromat as snm
#  create a network for the NOR Gate
XOR_Gate=snm.SNN()
### Run this code to see the parameters of a neuron in SNM ###
help(XOR_Gate.create_neuron)

Help on method create_neuron in module superneuromat.neuromorphicmodel:

create_neuron(
    threshold: float = 0.0,
    leak: float = inf,
    reset_state: float = 0.0,
    refractory_period: int = 0,
    refractory_state: int = 0,
    initial_state: float | None = 0.0
) -> Neuron method of superneuromat.neuromorphicmodel.SNN instance
    Create a neuron in the SNN.

    Parameters
    ----------
    threshold : float, default=0.0
        Neuron threshold; the neuron spikes if its internal state is strictly
        greater than the neuron threshold
    leak : float, default=numpy.inf
        Neuron leak; the amount by which the internal state of the neuron is
        pushed towards its reset state
    reset_state : float, default=0.0
        Reset state of the neuron; the value assigned to the internal state
        of the neuron after spiking
    refractory_period : int, default=0
        Refractory period of the neuron; the number of time steps for which
        the neuron remains in

In our creation of this gate, we will utilize thresholds, delay, and weights to determine the function of the system. For this project, we will have two input neurons and one output neuron, along with a neuron that will cancel the output if both neurons activate. The input neurons will receive either a 0 or 1 from the user, with a 1 denoting a "spike". They will then communicate their information via "synapses" to the output neuron.
<br>For the XOR Gate, we will need 4 total neurons and three synapses. We will start creating the neurons as the next step. The input neurons need a threshold of 0, as they should spike no matter what input they receive. The XOR Gate should spike if only one input neuron spikes. 

<br>![diagram of XOR_gate neurons](img/logic_gates_images/xor_gate_neurons.png "XOR_gate")

Here is a diagram of the neurons, their thresholds, and their connections. The numbers indicate the indices of specific neurons and the names designate neurons with special uses.

## STEP 1: Create Neurons

In [43]:
inputs=[]
outputs=[]
###First, we create the two input neurons with threshold 0 and add them to the inputs list
for i in range(2):
    id=XOR_Gate.create_neuron(threshold=0)
    inputs.append(id)


###Next, we create the final output neuron with threshold 0 and adds it to the outputs list
id=XOR_Gate.create_neuron(threshold=0)
outputs.append(id)

###The cancel neuron needs a threshold of 1 so it will spike if both input neurons also spike.
cancel=XOR_Gate.create_neuron(threshold=1)

###Finally, we simulate the neural network and print it
print(XOR_Gate)



SNN with 4 neurons and 0 synapses @ 0x1daff6caf90
STDP is globally enabled
apos: []
aneg: []
0 synapses have STDP enabled.

Neuron Info (4):
   idx       state      thresh        leak  ref_state  ref_period spikes
     0           0           0         inf          0           0 []
     1           0           0         inf          0           0 []
     2           0           0         inf          0           0 []
     3           0           1         inf          0           0 []

Synapse Info (0):
    idx    pre ->   post      weight    delay stdp_enabled


Input Spikes (0) for 0 time steps:
 Time:  Spike-value    Destination


Spike Train:
 t|id 
0 spikes since last reset


## STEP 2: Create Synapses

Next, we will take care of the synapses. For each synapse, we need to indicate the sending neuron, the receiving neuron, and the weight of the connection. We need to delay the input neurons so that our middle layer, the cancel neuron, can activate if both neurons spike. The cancel neuron needs a negative weight to fulfill its function of negating the internal state of the output neuron.

In [44]:
for i in range(2):
    #Creates synapses between the inputs and the output
    XOR_Gate.create_synapse(inputs[i],outputs[0],1,delay=2)
    XOR_Gate.create_synapse(inputs[i],cancel,1)

#Creates a synapse between the cancel neuron and the output
XOR_Gate.create_synapse(cancel,outputs[0],weight=-2)


print(XOR_Gate)


SNN with 6 neurons and 7 synapses @ 0x1daff6caf90
STDP is globally enabled
apos: []
aneg: []
0 synapses have STDP enabled.

Neuron Info (6):
   idx       state      thresh        leak  ref_state  ref_period spikes
     0           0           0         inf          0           0 []
     1           0           0         inf          0           0 []
     2           0           0         inf          0           0 []
     3           0           1         inf          0           0 []
     4           0           0         inf          0           0 []
     5           0           0         inf          0           0 []

Synapse Info (7):
    idx    pre ->   post      weight    delay stdp_enabled
      0      0 ->      4           1       1 -
      1      4 ->      2           1     - 2 -
      2      0 ->      3           1       1 -
      3      1 ->      5           1       1 -
      4      5 ->      2           1     - 2 -
      5      1 ->      3           1       1 -
      6     

## STEP 3: Add Spikes

Finally, we need to add the spikes. These spikes will be 1's. As we are working on the NOR gate, if neither neuron spikes, the output neuron will spike. To add the spikes, we specify time step, neuron to be spiked, and the value of the spike.  Once the spikes for each combination of two inputs are added, we need to simulate the program to see its function. As each transfer from input neuron to output neuron takes 2 times steps, we need to simulate 12 total time steps (starting at 0, gives us simulate(12)) to see every combination of inputs and outputs for the program.
<br>To see the results, look at the first three neurons in the spike train. The first two (indices 0,1) are the inputs and the third (index 2) is the output. This train should match the truth table for the XOR Gate from the beginning of this tutorial

In [45]:
XOR_Gate.add_spike(0,inputs[0],0)
XOR_Gate.add_spike(0,inputs[1],0)

XOR_Gate.add_spike(3,inputs[0],1)
XOR_Gate.add_spike(3,inputs[1],0)

XOR_Gate.add_spike(6,inputs[0],0)
XOR_Gate.add_spike(6,inputs[1],1)

XOR_Gate.add_spike(9,inputs[0],1)
XOR_Gate.add_spike(9,inputs[1],1)

XOR_Gate.simulate(12) 
print(XOR_Gate)


SNN with 6 neurons and 7 synapses @ 0x1daff6caf90
STDP is globally enabled
apos: []
aneg: []
0 synapses have STDP enabled.

Neuron Info (6):
   idx       state      thresh        leak  ref_state  ref_period spikes
     0           0           0         inf          0           0 [---┴⋯---┴--]
     1           0           0         inf          0           0 [----⋯┴--┴--]
     2           0           0         inf          0           0 [----⋯--┴---]
     3           0           1         inf          0           0 [----⋯----┴-]
     4           0           0         inf          0           0 [----⋯----┴-]
     5           0           0         inf          0           0 [----⋯-┴--┴-]

Synapse Info (7):
    idx    pre ->   post      weight    delay stdp_enabled
      0      0 ->      4           1       1 -
      1      4 ->      2           1     - 2 -
      2      0 ->      3           1       1 -
      3      1 ->      5           1       1 -
      4      5 ->      2           1    

If you have not already done so, try to create the NAND Gate on your own. Then, try the NAND gate tutorial, also within this GitHub, to see the solution.

In [46]:
###Put your code below